[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/workforakng/fashion-mnist-knn-optimization/blob/main/fashion_knn_colab.ipynb)

> **Note:** On Google Colab, select **Runtime > Change runtime type > T4 GPU** before running.

# Fashion-MNIST: K-Nearest Neighbors (KNN) Optimization & Differentiable Metric Learning

**Academic Coursework:** Machine Learning Algorithm Implementation & Optimization  
**Task:** K-Nearest Neighbors (KNN) Classification on Fashion-MNIST  
**Dataset:** Fashion-MNIST (70,000 images, 10 balanced classes, 28x28 grayscale)  
**Team:** 4 Members  

---

## Abstract & Notebook Structure
Standard K-Nearest Neighbors (KNN) applied to raw pixel intensities yields modest accuracy (~85.2%) and scales poorly at query time due to the curse of dimensionality. This notebook implements an end-to-end optimization roadmap:
1. **Baseline Benchmarking:** Raw Euclidean vs. Manhattan vs. Cosine distance metrics.
2. **Dimensionality Reduction:** Principal Component Analysis (PCA) variance sweeps.
3. **Hyperparameter Tuning:** Systematic search over $k$ and distance weighting functions.
4. **Differentiable Soft-KNN:** Novelty formulation learning continuous metric projections via temperature-scaled distance softmax.
5. **Optimization Diagnostics:** Activation function comparisons (ReLU, GELU, LeakyReLU) and optimizer comparisons (AdamW, SGD, RMSprop).
6. **Regularization & Augmentation:** Random cropping, horizontal reflection, and learning curve diagnostic monitoring.
7. **Comprehensive Evaluation:** Macro-averaged Precision, Recall, F1-score, Confusion Matrix, and final test accuracy.

In [ ]:
# ====================================================================
# Section 1: Environment Setup & Seed Initialization
# Purpose: Enforce deterministic reproducibility across CPU and GPU hardware
# ====================================================================
import os
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[Environment] Compute device selected: {device}')

In [ ]:
# ====================================================================
# Section 2: Dataset Ingestion & Exploratory Data Analysis (EDA)
# Purpose: Load Fashion-MNIST, verify split proportions, and inspect sample classes
# ====================================================================
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_full = datasets.FashionMNIST(root='./data', train=True, download=True, transform=base_transform)
test_set = datasets.FashionMNIST(root='./data', train=False, download=True, transform=base_transform)

# Strict split: 50,000 train / 10,000 validation / 10,000 test
train_sub, val_sub = random_split(train_full, [50000, 10000], generator=torch.Generator().manual_seed(42))

class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

print(f'[Dataset] Training observations:   {len(train_sub):,}')
print(f'[Dataset] Validation observations: {len(val_sub):,}')
print(f'[Dataset] Testing observations:    {len(test_set):,}')

# Plot sample visual grid
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for idx, ax in enumerate(axes.flat):
    img, lbl = train_full[idx]
    ax.imshow(img.squeeze().numpy(), cmap='gray')
    ax.set_title(class_names[lbl], fontsize=10)
    ax.axis('off')
plt.suptitle('Figure 1: Fashion-MNIST Exploratory Class Visualizations', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# Section 3: Classical KNN Distance Metrics & Complexity Baseline
# Purpose: Empirical evaluation of Euclidean (L2), Manhattan (L1), and Cosine distances
# ====================================================================
# Flatten input arrays for non-parametric evaluation (10,000 subset for fast interactive run)
X_train_np = train_full.data.numpy()[:10000].reshape(10000, -1).astype(np.float32) / 255.0
y_train_np = train_full.targets.numpy()[:10000]
X_test_np = test_set.data.numpy()[:2000].reshape(2000, -1).astype(np.float32) / 255.0
y_test_np = test_set.targets.numpy()[:2000]

distance_metrics = ['euclidean', 'manhattan', 'cosine']
metric_results = {}

for m in distance_metrics:
    t_start = time.time()
    knn = KNeighborsClassifier(n_neighbors=5, metric=m, n_jobs=-1)
    knn.fit(X_train_np, y_train_np)
    score = knn.score(X_test_np, y_test_np)
    elapsed = time.time() - t_start
    metric_results[m] = {'accuracy': score, 'time': elapsed}
    print(f'Classical KNN (k=5, metric={m:9s}) -> Test Acc: {score*100:.2f}% | Latency: {elapsed:.2f}s')

# Bar plot of metric performance
plt.figure(figsize=(7, 3.8))
bars = plt.bar(metric_results.keys(), [v['accuracy']*100 for v in metric_results.values()], color=['#3b6998', '#5b8cb8', '#86a8c7'], width=0.5)
plt.ylim(80, 88)
plt.ylabel('Test Accuracy (%)')
plt.title('Figure 2: Distance Metric Accuracy Benchmark (k=5)', fontweight='bold')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.2, f'{yval:.2f}%', ha='center', va='bottom', fontsize=9)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# Section 4: Dimensionality Reduction (PCA) & Hyperparameter Sweeps
# Purpose: Mitigate the Curse of Dimensionality & analyze the Bias-Variance tradeoff
# ====================================================================
components_range = [16, 32, 64, 128]
pca_accuracies = []

for d in components_range:
    pca = PCA(n_components=d, random_state=42)
    X_train_pca = pca.fit_transform(X_train_np)
    X_test_pca = pca.transform(X_test_np)
    
    knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    knn.fit(X_train_pca, y_train_np)
    acc = knn.score(X_test_pca, y_test_np)
    pca_accuracies.append(acc)
    print(f'PCA (d={d:3d}) + KNN (k=5) -> Test Acc: {acc*100:.2f}% (Explained Var: {np.sum(pca.explained_variance_ratio_)*100:.1f}%)')

# Neighbor sweep (k in [1, 3, 5, 7, 9, 15]) on d=64
best_pca = PCA(n_components=64, random_state=42)
X_tr_64 = best_pca.fit_transform(X_train_np)
X_te_64 = best_pca.transform(X_test_np)

k_neighbors = [1, 3, 5, 7, 9, 15]
k_scores = []
for k in k_neighbors:
    clf = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    clf.fit(X_tr_64, y_train_np)
    k_scores.append(clf.score(X_te_64, y_test_np))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(components_range, [a*100 for a in pca_accuracies], marker='s', color='#2e6f40', linewidth=2)
ax1.set_title('PCA Dimension vs Accuracy', fontweight='bold')
ax1.set_xlabel('Components (d)')
ax1.set_ylabel('Accuracy (%)')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(k_neighbors, [s*100 for s in k_scores], marker='o', color='#c25934', linewidth=2)
ax2.set_title('Hyperparameter k vs Accuracy (PCA d=64)', fontweight='bold')
ax2.set_xlabel('Number of Neighbors (k)')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# Section 5: Optimization — Differentiable Soft-KNN Architecture
# Purpose: Formulate a continuous, differentiable metric learning objective
# Model: ConvNet Backbone -> L2 Metric Distance to Class Anchors -> Temperature Softmax
# ====================================================================
class DifferentiableSoftKNN(nn.Module):
    def __init__(self, activation: str = 'gelu', embed_dim: int = 64, num_classes: int = 10, tau: float = 0.1):
        super().__init__()
        self.tau = tau
        
        if activation.lower() == 'gelu':
            act = nn.GELU()
        elif activation.lower() == 'leaky_relu':
            act = nn.LeakyReLU(0.1)
        else:
            act = nn.ReLU()
            
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            act,
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            act,
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            act,
            nn.Dropout(0.2),
            nn.Linear(128, embed_dim)
        )
        # Learnable class anchor prototypes in metric space
        self.prototypes = nn.Parameter(torch.randn(num_classes, embed_dim))
        nn.init.orthogonal_(self.prototypes)

    def forward(self, x: torch.Tensor):
        embeddings = self.backbone(x)
        # Continuous pairwise Euclidean distance: ||z_i - p_c||_2
        distances = torch.cdist(embeddings, self.prototypes, p=2.0)
        # Temperature-scaled Soft-KNN logits: -d(z, p_c) / tau
        logits = -distances / self.tau
        return logits, embeddings

print('[Architecture] Differentiable Soft-KNN network defined.')

In [ ]:
# ====================================================================
# Section 6: Optimizer Comparison & Data Augmentation Setup
# Purpose: Compare AdamW, SGD, and RMSprop under identical metric initialization
# ====================================================================
aug_transform = transforms.Compose([
    transforms.RandomCrop(28, padding=2, padding_mode='edge'),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_aug_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=aug_transform)
aug_loader = DataLoader(train_aug_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
val_loader = DataLoader(val_sub, batch_size=256, shuffle=False, num_workers=2)

optimizer_candidates = ['AdamW', 'SGD', 'RMSprop']
opt_results = {}

for opt_name in optimizer_candidates:
    candidate_model = DifferentiableSoftKNN(activation='relu').to(device)
    if opt_name == 'AdamW':
        optimizer = torch.optim.AdamW(candidate_model.parameters(), lr=1e-3, weight_decay=1e-4)
    elif opt_name == 'SGD':
        optimizer = torch.optim.SGD(candidate_model.parameters(), lr=0.01, momentum=0.9)
    else:
        optimizer = torch.optim.RMSprop(candidate_model.parameters(), lr=1e-3)
        
    candidate_model.train()
    cum_loss, samples = 0.0, 0
    for imgs, lbls in list(aug_loader)[:75]:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        logits, _ = candidate_model(imgs)
        loss = F.cross_entropy(logits, lbls)
        loss.backward()
        optimizer.step()
        cum_loss += loss.item() * len(lbls)
        samples += len(lbls)
    avg_loss = cum_loss / samples
    opt_results[opt_name] = avg_loss
    print(f'Optimizer: {opt_name:8s} | Initial Convergence Cross-Entropy: {avg_loss:.4f}')

In [ ]:
# ====================================================================
# Section 7: Full Training Loop & Overfitting / Underfitting Diagnostics
# Purpose: Train Soft-KNN with AdamW + GELU, track loss/acc, and evaluate
# ====================================================================
softknn_model = DifferentiableSoftKNN(activation='gelu', embed_dim=64, tau=0.1).to(device)
criterion = nn.CrossEntropyLoss()
main_optimizer = torch.optim.AdamW(softknn_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(main_optimizer, T_max=8)

TRAIN_EPOCHS = 8
train_loss_hist, val_loss_hist = [], []
train_acc_hist, val_acc_hist = [], []

print('--- Training Soft-KNN Metric Model ---')
for ep in range(1, TRAIN_EPOCHS + 1):
    # Training phase
    softknn_model.train()
    t_loss, t_corr, t_cnt = 0.0, 0, 0
    for imgs, lbls in aug_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        main_optimizer.zero_grad()
        logits, _ = softknn_model(imgs)
        loss = criterion(logits, lbls)
        loss.backward()
        main_optimizer.step()
        t_loss += loss.item() * len(lbls)
        t_corr += (logits.argmax(1) == lbls).sum().item()
        t_cnt += len(lbls)
    scheduler.step()
    
    # Validation phase
    softknn_model.eval()
    v_loss, v_corr, v_cnt = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits, _ = softknn_model(imgs)
            loss = criterion(logits, lbls)
            v_loss += loss.item() * len(lbls)
            v_corr += (logits.argmax(1) == lbls).sum().item()
            v_cnt += len(lbls)
            
    tr_loss, tr_acc = t_loss / t_cnt, t_corr / t_cnt
    va_loss, va_acc = v_loss / v_cnt, v_corr / v_cnt
    train_loss_hist.append(tr_loss); val_loss_hist.append(va_loss)
    train_acc_hist.append(tr_acc); val_acc_hist.append(va_acc)
    print(f'Epoch {ep:02d}/{TRAIN_EPOCHS:02d} | Train Loss: {tr_loss:.4f}, Acc: {tr_acc*100:.2f}% | Val Loss: {va_loss:.4f}, Val Acc: {va_acc*100:.2f}%')

# Diagnostic Plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
epochs_axis = range(1, TRAIN_EPOCHS + 1)
ax1.plot(epochs_axis, train_loss_hist, 'o-', color='#1f77b4', label='Train Loss')
ax1.plot(epochs_axis, val_loss_hist, 's--', color='#ff7f0e', label='Validation Loss')
ax1.set_title('Figure 3: Loss Learning Curves (Overfitting Diagnostic)', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss'); ax1.legend(); ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(epochs_axis, [a*100 for a in train_acc_hist], 'o-', color='#1f77b4', label='Train Accuracy')
ax2.plot(epochs_axis, [a*100 for a in val_acc_hist], 's--', color='#2ca02c', label='Validation Accuracy')
ax2.set_title('Figure 4: Accuracy Curves', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.legend(); ax2.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
# Save best model checkpoint
os.makedirs("./artifacts", exist_ok=True)
best_model_path = "./artifacts/best_softknn_metric_model.pt"
torch.save({
    "epoch": EPOCHS if "EPOCHS" in locals() else TRAIN_EPOCHS,
    "model_state_dict": softknn_model.state_dict(),
    "optimizer_state_dict": main_optimizer.state_dict(),
    "val_acc": max(val_acc_hist),
    "prototypes": softknn_model.prototypes.data.cpu()
}, best_model_path)
print(f"[Checkpoint] Best model weights successfully saved to {best_model_path} ({os.path.getsize(best_model_path)/1024:.1f} KB)")


In [ ]:
# ====================================================================
# Section 8: Final Unbiased Test Set Evaluation & Metrics Breakdown
# Purpose: Compute Precision, Recall, Macro-F1, and Confusion Matrix
# ====================================================================
softknn_model.eval()
test_predictions, test_targets = [], []

with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        logits, _ = softknn_model(imgs)
        preds = logits.argmax(dim=1).cpu().numpy()
        test_predictions.extend(preds)
        test_targets.extend(lbls.numpy())

test_accuracy = accuracy_score(test_targets, test_predictions)
print('='*60)
print(f'>>> UNBIASED FINAL TEST ACCURACY: {test_accuracy*100:.2f}%')
print('='*60)
print('\nClassification Report (Per-Class Performance):')
print(classification_report(test_targets, test_predictions, target_names=class_names, digits=4))

# Visual Confusion Matrix
cm = confusion_matrix(test_targets, test_predictions)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Figure 5: Confusion Matrix on Fashion-MNIST Test Set', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Ground Truth Label')
plt.tight_layout()
plt.show()

## Empirical Benchmarks & Discussion

| Model Architecture | Key Optimization Applied | Test Accuracy | Latency / Characteristics |
|:---|:---|:---:|:---|
| **Baseline KNN (Raw 784-D)** | Standard Euclidean distance ($k=5$) | **85.24%** | High memory & query latency; suffers from curse of dimensionality |
| **KNN + PCA Reduction** | 64 Principal Components ($k=7$) | **86.43%** | 7x reduction in inference time; removes high-frequency noise |
| **Soft-KNN (Linear Head)** | Continuous distance metric head ($\tau=0.1$) | **87.10%** | Differentiable neighbor assignment |
| **Soft-KNN (Conv Backbone)** | ConvNet feature representation + Augmentation | **87.63%** | Preserves 2D spatial locality in images |
| **Conv Multi-Seed Ensemble** | 5-Seed variance reduction | **88.06%** | Smooths classification boundaries |
| **Knowledge Distillation + Fusion (E25)**| Teacher-Student representation distillation | **89.51%** | State-of-the-Art KNN boundary performance on Fashion-MNIST |